# Dataset loading 

In [1]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))

c:\Users\lalis\miniconda3\envs\ai_conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Tokenization

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14859.91it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

# Preprocess the dataset for fine tuning 

In [3]:
def preprocess(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(preprocess, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Apply Prompt Tuning (PEFT)

In [4]:
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model, TaskType

config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=20,
    prompt_tuning_init_text="Classify the sentiment of this movie review:",
    tokenizer_name_or_path=model_name,
)

peft_model = get_peft_model(model, config)

peft_model.print_trainable_parameters()

trainable params: 16,898 || all params: 109,500,676 || trainable%: 0.0154


# Training args

In [5]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./prompt-tuning-imdb",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

In [6]:
trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,0.688220,0.665390
2,0.661559,0.644403
3,0.649929,0.637596


TrainOutput(global_step=1875, training_loss=0.6618065673828125, metrics={'train_runtime': 194.6542, 'train_samples_per_second': 154.119, 'train_steps_per_second': 9.632, 'total_flos': 3946736701440000.0, 'train_loss': 0.6618065673828125, 'epoch': 3.0})

# Inference

In [9]:
import torch
device = peft_model.device

text = "This movie was surprisingly good and emotional."
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = peft_model(**inputs)

pred = outputs.logits.argmax(dim=-1).item()
print("Positive" if pred == 1 else "Negative")

Positive
